In [ ]:
""" Results obtained from the following IPython notebooks:

- Gopi_TEC_Brasilia_analysis_09_27_2024.ipynb (BRAZ station)
- Gopi_TEC_Brasilia_analysis_Dec_2024.ipynb (BRAZ station)
- Gopi_TEC_Cuiaba_analysis_09_27_2024.ipynb (CUIB station)
- Gopi_TEC_Cuiaba_analysis_Dec_2024.ipynb (CUIB station)
- Gopi_TEC_Manaus_analysis_09_27_2024.ipynb (NAUS station)
- Gopi_TEC_Manaus_analysis_Dec_2024.ipynb (NAUS station)
- Gopi_TEC_Porto_Alegre_analysis_09_27_2024.ipynb (POAL station)
- Gopi_TEC_Porto_Alegre_analysis_Dec_2024.ipynb (POAL station)
- Gopi_TEC_SJC_analysis_09_27_2024.ipynb (SJSP station)
- Gopi_TEC_SJC_analysis_Dec_2024.ipynb (SJSP station)
- Gopi_TEC_Salvador_analysis_09_27_2024.ipynb (SAVO station)
- Gopi_TEC_Salvador_analysis_Dec_2024.ipynb (SAVO station)
- Gopi_TEC_Sao_Luis_analysis_09_27_2024.ipynb (SALU station)
- Gopi_TEC_Sao_Luis_analysis_Dec_2024.ipynb (SALU station) """

In [4]:
import numpy as np
from scipy import stats
import pandas as pd

def pearson_correlation(observed, predicted):
    """Calculate Pearson correlation without Fisher transformation"""
    correlation, p_value = stats.pearsonr(observed, predicted)
    return correlation, p_value

def taylor_skill_score(observed, predicted):
    observed = np.array(observed)
    predicted = np.array(predicted)

    correlation = np.corrcoef(observed, predicted)[0, 1]

    std_obs = np.std(observed, ddof=1)
    std_pred = np.std(predicted, ddof=1)

    if std_obs == 0 or std_pred == 0:
        return 0.0

    std_ratio = std_pred / std_obs

    denominator = (std_ratio + 1/std_ratio)**2 * (1 + 1.0)
    skill_score = (4 * (1 + correlation)) / denominator

    return skill_score

def kling_gupta_efficiency(observed, predicted):
    observed = np.array(observed)
    predicted = np.array(predicted)

    r = np.corrcoef(observed, predicted)[0, 1]

    std_obs = np.std(observed, ddof=1)
    std_pred = np.std(predicted, ddof=1)
    alpha = std_pred / std_obs if std_obs != 0 else np.inf

    mean_obs = np.mean(observed)
    mean_pred = np.mean(predicted)
    beta = mean_pred / mean_obs if mean_obs != 0 else np.inf

    kge = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)

    components = {'r': r, 'alpha': alpha, 'beta': beta}

    return kge, components

def comprehensive_ranking_index(metrics_dict, weights=None):
    if weights is None:
        weights = {
            'mae': 0.2,
            'rmse': 0.2,
            'correlation': 0.2,
            'tss': 0.2,
            'kge': 0.2
        }

    models = list(metrics_dict.keys())
    metrics = ['mae', 'rmse', 'correlation', 'tss', 'kge']

    normalized_metrics = {}

    for metric in metrics:
        values = [metrics_dict[model][metric] for model in models]
        min_val, max_val = min(values), max(values)

        if metric in ['mae', 'rmse']:
            normalized_metrics[metric] = {
                model: 1 - (metrics_dict[model][metric] - min_val) / (max_val - min_val)
                if max_val != min_val else 1
                for model in models
            }
        else:
            normalized_metrics[metric] = {
                model: (metrics_dict[model][metric] - min_val) / (max_val - min_val)
                if max_val != min_val else 1
                for model in models
            }

    cri_scores = {}
    for model in models:
        cri = sum(weights[metric] * normalized_metrics[metric][model] for metric in metrics)
        cri_scores[model] = cri

    rankings = {model: rank for rank, model in enumerate(
        sorted(models, key=lambda x: cri_scores[x], reverse=True), 1)}

    return cri_scores, rankings

def calculate_all_metrics(observed, predicted, model_name="Model"):
    """Calculate all metrics, properly handling NaN values"""
    observed = np.array(observed)
    predicted = np.array(predicted)

    # Verificar se temos dados suficientes
    if len(observed) < 2 or len(predicted) < 2:
        return {
            'model': model_name,
            'mae': np.nan,
            'rmse': np.nan,
            'correlation': np.nan,
            'correlation_p_value': np.nan,
            'taylor_skill_score': 0.0,
            'kge': 0.0,
            'kge_r': 0.0,
            'kge_alpha': 0.0,
            'kge_beta': 0.0
        }

    # Filtrar valores NaN (embora já tenhamos filtrado antes)
    mask = ~(np.isnan(observed) | np.isnan(predicted))
    observed_filtered = observed[mask]
    predicted_filtered = predicted[mask]

    if len(observed_filtered) < 2:
        return {
            'model': model_name,
            'mae': np.nan,
            'rmse': np.nan,
            'correlation': np.nan,
            'correlation_p_value': np.nan,
            'taylor_skill_score': 0.0,
            'kge': 0.0,
            'kge_r': 0.0,
            'kge_alpha': 0.0,
            'kge_beta': 0.0
        }

    diff = observed_filtered - predicted_filtered
    mae = np.mean(np.abs(diff))
    rmse = np.sqrt(np.mean(diff**2))

    correlation, p_value = pearson_correlation(observed_filtered, predicted_filtered)

    tss = taylor_skill_score(observed_filtered, predicted_filtered)

    kge, kge_components = kling_gupta_efficiency(observed_filtered, predicted_filtered)

    metrics = {
        'model': model_name,
        'mae': mae,
        'rmse': rmse,
        'correlation': correlation,
        'correlation_p_value': p_value,
        'taylor_skill_score': tss,
        'kge': kge,
        'kge_r': kge_components['r'],
        'kge_alpha': kge_components['alpha'],
        'kge_beta': kge_components['beta']
    }

    return metrics

def enhanced_analysis(data, reference_model='Gopi'):
    """Enhanced analysis function that properly handles NaN values"""
    # Dicionário para armazenar dados por modelo
    model_data_pairs = {}

    first_station = list(data.keys())[0]
    available_models = [model for model in data[first_station].keys() if model != reference_model]

    # Inicializar dicionário para cada modelo
    for model in available_models:
        model_data_pairs[model] = {
            'reference': [],  # Para armazenar valores do modelo de referência
            'model': []       # Para armazenar valores do modelo comparado
        }

    # Coletar pares de dados válidos para cada modelo
    for station in data.keys():
        station_ref_data = data[station][reference_model]

        for model in available_models:
            model_data = data[station][model]

            # Processar cada ponto de dados, adicionando apenas pares não-NaN
            for i in range(min(len(model_data), len(station_ref_data))):
                if not np.isnan(model_data[i]) and not np.isnan(station_ref_data[i]):
                    model_data_pairs[model]['reference'].append(station_ref_data[i])
                    model_data_pairs[model]['model'].append(model_data[i])

    # Calcular métricas para cada modelo usando seus próprios pares de referência
    all_metrics = {}
    for model in available_models:
        reference_values = np.array(model_data_pairs[model]['reference'])
        model_values = np.array(model_data_pairs[model]['model'])

        # Verificar se há dados suficientes para cálculo
        if len(reference_values) > 1:
            metrics = calculate_all_metrics(reference_values, model_values, model)
        else:
            # Criar métricas padrão se não houver dados suficientes
            metrics = {
                'model': model,
                'mae': np.nan,
                'rmse': np.nan,
                'correlation': np.nan,
                'correlation_p_value': np.nan,
                'taylor_skill_score': np.nan,
                'kge': np.nan,
                'kge_r': np.nan,
                'kge_alpha': np.nan,
                'kge_beta': np.nan
            }
        all_metrics[model] = metrics

    # Preparar métricas para CRI
    metrics_for_cri = {}
    for model in available_models:
        metrics_for_cri[model] = {
            'mae': all_metrics[model]['mae'],
            'rmse': all_metrics[model]['rmse'],
            'correlation': all_metrics[model]['correlation'],
            'tss': all_metrics[model]['taylor_skill_score'],
            'kge': all_metrics[model]['kge']
        }

    # Calcular CRI e rankings
    cri_scores, rankings = comprehensive_ranking_index(metrics_for_cri)

    # Adicionar CRI e rankings às métricas
    for model in available_models:
        all_metrics[model]['cri'] = cri_scores[model]
        all_metrics[model]['rank'] = rankings[model]

    return all_metrics

def print_enhanced_results(results, title="Enhanced Analysis Results"):
    print("=" * 80)
    print(f"{title}")
    print("=" * 80)

    sorted_models = sorted(results.keys(), key=lambda x: results[x]['rank'])

    for model in sorted_models:
        metrics = results[model]
        print(f"\n{model.upper()} (Rank #{metrics['rank']}):")
        print(f"  MAE:                   {metrics['mae']:.2f}")
        print(f"  RMSE:                  {metrics['rmse']:.2f}")
        print(f"  Pearson Correlation:   {metrics['correlation']:.3f} (p={metrics['correlation_p_value']:.3f})")
        print(f"  Taylor Skill Score:    {metrics['taylor_skill_score']:.3f}")
        print(f"  Kling-Gupta Efficiency: {metrics['kge']:.3f}")
        print(f"    - Correlation (r):   {metrics['kge_r']:.3f}")
        print(f"    - Alpha (σp/σo):     {metrics['kge_alpha']:.3f}")
        print(f"    - Beta (μp/μo):      {metrics['kge_beta']:.3f}")
        print(f"  CRI Score:             {metrics['cri']:.3f}")

    print("\n" + "=" * 80)
    print("RANKING SUMMARY:")
    for model in sorted_models:
        print(f"  {results[model]['rank']}. {model.upper()} (CRI: {results[model]['cri']:.3f})")
    print("=" * 80)

# September

In [ ]:
#The values ​​below refer, for each TEC source, consecutively, to the following times for September 27, 2024: 00:50 UT 01:00 UT 01:10 UT.

In [ ]:
import numpy as np

data = {
    'BRAZ': {
        'Gopi': [91.93, 92.28, 93.38],
        'EMBRACE': [72.52, 71.86, 67.75],
        'MAGGIA': [138.74, 133.04, 129.11]
    },
    'CUIB': {
        'Gopi': [73.90, 73.87, 74.22],
        'EMBRACE': [36.89, 43.34, 42.93],
        'MAGGIA': [75.94, 111.90, 85.23]
    },
    'NAUS': {
        'Gopi': [19.83, 19.68, 19.47],
        'EMBRACE': [28.84, 32.11, 34.07],
        'MAGGIA': [40.93, 39.25, 44.92]
    },
    'POAL': {
        'Gopi': [14.77, 15.80, 16.92],
        'EMBRACE': [33.90, 33.01, 34.13],
        'MAGGIA': [29.80, 27.86, 26.85]
    },
    'SALU': {
        'Gopi': [21.98, 20.75, 19.45],
        'EMBRACE': [39.31, 41.23, 34.59],
        'MAGGIA': [38.69, 42.25, 43.50]
    },
    'SAVO': {
        'Gopi': [101.46, 98.65, 94.63],
        'EMBRACE': [69.32, 73.81, 73.59],
        'MAGGIA': [121.90, 133.17, 127.19]
    },
    'SJSP': {
        'Gopi': [30.89, 30.06, 29.8],
        'EMBRACE': [31.64, 30.80, 28.02],
        'MAGGIA': [51.12, 49.78, 47.96]
    }
}

gopi_all = []
embrace_all = []
maggia_all = []

for station in data.keys():
    gopi_all.extend(data[station]['Gopi'])
    embrace_all.extend(data[station]['EMBRACE'])
    maggia_all.extend(data[station]['MAGGIA'])

gopi_all = np.array(gopi_all)
embrace_all = np.array(embrace_all)
maggia_all = np.array(maggia_all)

diff_embrace = gopi_all - embrace_all
diff_maggia = gopi_all - maggia_all

mae_embrace = np.mean(np.abs(diff_embrace))
mae_maggia = np.mean(np.abs(diff_maggia))

rmse_embrace = np.sqrt(np.mean(diff_embrace**2))
rmse_maggia = np.sqrt(np.mean(diff_maggia**2))

print("="*60)
print("CONSOLIDATED RESULTS - September 27, 2024")
print("(21 measurements: 7 stations × 3 times)")
print("="*60)
print(f"GOPI vs EMBRACE:")
print(f"  MAE:  {mae_embrace:.2f}")
print(f"  RMSE: {rmse_embrace:.2f}")
print()
print(f"GOPI vs MAGGIA:")
print(f"  MAE:  {mae_maggia:.2f}")
print(f"  RMSE: {rmse_maggia:.2f}")

results_september = enhanced_analysis(data)
print_enhanced_results(results_september, "COMPREHENSIVE ANALYSIS - September 27, 2024")

CONSOLIDATED RESULTS - September 27, 2024
(21 measurements: 7 stations × 3 times)
GOPI vs EMBRACE:
  MAE:  18.48
  RMSE: 20.91

GOPI vs MAGGIA:
  MAE:  23.11
  RMSE: 25.60
COMPREHENSIVE ANALYSIS - September 27, 2024

MAGGIA (Rank #1):
  MAE:                   23.11
  RMSE:                  25.60
  Pearson Correlation:   0.974 (p=0.000)
  Taylor Skill Score:    0.953
  Kling-Gupta Efficiency: 0.494
    - Correlation (r):   0.974
    - Alpha (σp/σo):     1.209
    - Beta (μp/μo):      1.461
  CRI Score:             0.600

EMBRACE (Rank #2):
  MAE:                   18.48
  RMSE:                  20.91
  Pearson Correlation:   0.889 (p=0.000)
  Taylor Skill Score:    0.606
  Kling-Gupta Efficiency: 0.480
    - Correlation (r):   0.889
    - Alpha (σp/σo):     0.501
    - Beta (μp/μo):      0.905
  CRI Score:             0.400

RANKING SUMMARY:
  1. MAGGIA (CRI: 0.600)
  2. EMBRACE (CRI: 0.400)


# December

In [ ]:
#The values ​​below refer, for each TEC source, consecutively, to the following times for December 18, 2024: 18:20 UT, 18:10 UT, 18:30 UT and 18:00 UT.

In [6]:
import numpy as np

data = {
    'BRAZ': {
        'Gopi': [81.52, 80.90, 81.67, 79.74],
        'EMBRACE': [86.12, 82.63, 89.26, 81.36],
        'MAGGIA': [101.97, 100.54, 97.66, 99.78],
        'Nagoya': [86.32, 77.58, 82.50, 82.49]
    },
    'CUIB': {
        'Gopi': [76.57, 75.38, 77.54, 74.03],
        'EMBRACE': [78.75, 78.63, 77.95, 76.98],
        'MAGGIA': [98.65, 96.70, 85.99, 94.45],
        'Nagoya': [79.33, 77.14, 80.47, 75.42]
    },
    'NAUS': {
        'Gopi': [67.05, 67.10, 67.06, 67.33],
        'EMBRACE': [62.59, 62.57, 62.60, 63.07],
        'MAGGIA': [89.51, 94.99, 87.51, 87.40],
        'Nagoya': [60.88, 61.02, 61.64, 61.57]
    },
    'POAL': {
        'Gopi': [56.66, 57.57, 55.93, 58.40],
        'EMBRACE': [77.60, 76.34, 74.33, 75.83],
        'MAGGIA': [74.63, 75.28, 75.09, 75.03],
        'Nagoya': [77.51, 76.41, 78.36, 74.34]
    },
    'SALU': {
        'Gopi': [64.24, 64.09, 64.72, 64.24],
        'EMBRACE': [64.81, 65.81, 64.78, 66.44],
        'MAGGIA': [79.49, 79.40, 77.81, 79.03],
        'Nagoya': [67.51, 66.66, 68.84, 66.20]
    },
    'SAVO': {
        'Gopi': [89.57, 90.04, 88.68, 90.05],
        'EMBRACE': [83.02, 83.43, 84.79, 83.04],
        'MAGGIA': [98.48, 98.79, 101.57, 99.74],
        'Nagoya': [79.04, 77.60, 78.62, 83.50]
    },
    'SJSP': {
        'Gopi': [73.49, 73.19, 72.30, 71.73],
        'EMBRACE': [80.31, 80.79, 82.70, 80.31],
        'MAGGIA': [84.96, 84.76, 84.29, 83.35],
        'Nagoya': [81.58, 83.87, 82.26, 83.00]
    }
}

gopi_all = []
nagoya_all = []
embrace_all = []
maggia_all = []

for station in data.keys():
    gopi_all.extend(data[station]['Gopi'])
    nagoya_all.extend(data[station]['Nagoya'])
    embrace_all.extend(data[station]['EMBRACE'])
    maggia_all.extend(data[station]['MAGGIA'])

gopi_all = np.array(gopi_all)
nagoya_all = np.array(nagoya_all)
embrace_all = np.array(embrace_all)
maggia_all = np.array(maggia_all)

diff_nagoya = gopi_all - nagoya_all
diff_embrace = gopi_all - embrace_all
diff_maggia = gopi_all - maggia_all

mae_nagoya = np.mean(np.abs(diff_nagoya))
mae_embrace = np.mean(np.abs(diff_embrace))
mae_maggia = np.mean(np.abs(diff_maggia))

rmse_nagoya = np.sqrt(np.mean(diff_nagoya**2))
rmse_embrace = np.sqrt(np.mean(diff_embrace**2))
rmse_maggia = np.sqrt(np.mean(diff_maggia**2))

print("="*70)
print("CONSOLIDATED RESULTS - December 18, 2024")
print("(28 measurements: 7 stations × 4 times)")
print("="*70)
print(f"GOPI vs EMBRACE:")
print(f"  MAE:  {mae_embrace:.2f}")
print(f"  RMSE: {rmse_embrace:.2f}")
print()
print(f"GOPI vs MAGGIA:")
print(f"  MAE:  {mae_maggia:.2f}")
print(f"  RMSE: {rmse_maggia:.2f}")
print()
print(f"GOPI vs NAGOYA:")
print(f"  MAE:  {mae_nagoya:.2f}")
print(f"  RMSE: {rmse_nagoya:.2f}")
print("="*70)

results_december = enhanced_analysis(data)
print_enhanced_results(results_december, "COMPREHENSIVE ANALYSIS - December 18, 2024")

CONSOLIDATED RESULTS - December 18, 2024
(28 measurements: 7 stations × 4 times)
GOPI vs EMBRACE:
  MAE:  6.41
  RMSE: 8.59

GOPI vs MAGGIA:
  MAE:  16.29
  RMSE: 17.00

GOPI vs NAGOYA:
  MAE:  7.63
  RMSE: 9.64
COMPREHENSIVE ANALYSIS - December 18, 2024

EMBRACE (Rank #1):
  MAE:                   6.41
  RMSE:                  8.59
  Pearson Correlation:   0.645 (p=0.000)
  Taylor Skill Score:    0.787
  Kling-Gupta Efficiency: 0.595
    - Correlation (r):   0.645
    - Alpha (σp/σo):     0.810
    - Beta (μp/μo):      1.047
  CRI Score:             0.664

MAGGIA (Rank #2):
  MAE:                   16.29
  RMSE:                  17.00
  Pearson Correlation:   0.877 (p=0.000)
  Taylor Skill Score:    0.934
  Kling-Gupta Efficiency: 0.735
    - Correlation (r):   0.877
    - Alpha (σp/σo):     0.931
    - Beta (μp/μo):      1.225
  CRI Score:             0.600

NAGOYA (Rank #3):
  MAE:                   7.63
  RMSE:                  9.64
  Pearson Correlation:   0.491 (p=0.008)
  Taylor

In [ ]:
#The values ​​below refer, for each TEC source, consecutively, to the following times for December 18, 22, 19 and 1, 2024: 18:00 UT.

In [9]:
import numpy as np

data = {
    'BRAZ': {
        'Gopi': [79.74, 78.26, 72.14, 71.17],
        'EMBRACE': [81.36, 76.91, 61.01, 71.76],
        'IGS': [84.50, 84.55, 80.95, 87.98],
        'MAGGIA': [99.78, 94.35, 80.46, 85.46],
        'Nagoya': [82.50, 75.63, 67.53, 79.03]
    },
    'CUIB': {
        'Gopi': [74.03, 75.19, 68.55, 74.68],
        'EMBRACE': [76.98, 68.62, 65.20, 65.60],
        'IGS': [85.01, 85.99, 82.91, 89.13],
        'MAGGIA': [94.45, 94.83, 82.78, 100.93],
        'Nagoya': [75.42, 70.67, 66.21, 65.51]
    },
    'NAUS': {
        'Gopi': [67.33, 66.96, 62.93, 79.52],
        'EMBRACE': [63.07, 67.53, 62.12, 79.02],
        'IGS': [68.51, 72.48, 74.49, 69.08],
        'MAGGIA': [87.40, 93.03, 82.94, 100.49],
        'Nagoya': [61.57, 66.47, 64.89, np.nan]
    },
    'POAL': {
        'Gopi': [58.40, 60.67, 61.57, 59.27],
        'EMBRACE': [75.83, 75.73, 73.31, 74.77],
        'IGS': [87.51, 87.91, 79.46, 97.33],
        'MAGGIA': [75.03, 82.55, 80.20, 84.14],
        'Nagoya': [74.34, 76.95, 73.45, 75.83]
    },
    'SALU': {
        'Gopi': [64.24, 60.04, 57.14, 74.54],
        'EMBRACE': [66.44, 61.26, 59.13, 80.89],
        'IGS': [59.82, 65.04, 65.25, 65.48],
        'MAGGIA': [79.03, 81.07, 72.46, 98.98],
        'Nagoya': [66.20, 62.33, 59.06, 58.45]
    },
    'SAVO': {
        'Gopi': [90.05, 86.30, 72.09, 73.75],
        'EMBRACE': [83.04, 74.56, 75.58, 76.41],
        'IGS': [72.85, 75.92, 73.89, 81.44],
        'MAGGIA': [99.74, 99.28, 84.99, 95.67],
        'Nagoya': [83.50, 72.88, 70.32, -8.12]
    },
    'SJSP': {
        'Gopi': [71.73, 67.49, 67.35, 67.11],
        'EMBRACE': [80.30, 78.06, 76.98, 78.24],
        'IGS': [85.41, 85.00, 80.77, 88.07],
        'MAGGIA': [83.34, 87.43, 80.46, 88.18],
        'Nagoya': [83.00, 80.96, 76.41, 78.57]
    }
}

gopi_embrace = []
embrace_all = []
gopi_maggia = []
maggia_all = []
gopi_nagoya = []
nagoya_all = []
gopi_igs = []
igs_all = []

for station in data.keys():
    for i in range(len(data[station]['Gopi'])):
        gopi_val = data[station]['Gopi'][i]

        if i < len(data[station]['EMBRACE']) and not np.isnan(gopi_val) and not np.isnan(data[station]['EMBRACE'][i]):
            gopi_embrace.append(gopi_val)
            embrace_all.append(data[station]['EMBRACE'][i])

        if i < len(data[station]['MAGGIA']) and not np.isnan(gopi_val) and not np.isnan(data[station]['MAGGIA'][i]):
            gopi_maggia.append(gopi_val)
            maggia_all.append(data[station]['MAGGIA'][i])

        if i < len(data[station]['Nagoya']) and not np.isnan(gopi_val) and not np.isnan(data[station]['Nagoya'][i]):
            gopi_nagoya.append(gopi_val)
            nagoya_all.append(data[station]['Nagoya'][i])

        if i < len(data[station]['IGS']) and not np.isnan(gopi_val) and not np.isnan(data[station]['IGS'][i]):
            gopi_igs.append(gopi_val)
            igs_all.append(data[station]['IGS'][i])

gopi_embrace = np.array(gopi_embrace)
embrace_all = np.array(embrace_all)
gopi_maggia = np.array(gopi_maggia)
maggia_all = np.array(maggia_all)
gopi_nagoya = np.array(gopi_nagoya)
nagoya_all = np.array(nagoya_all)
gopi_igs = np.array(gopi_igs)
igs_all = np.array(igs_all)

diff_embrace = gopi_embrace - embrace_all
diff_maggia = gopi_maggia - maggia_all
diff_nagoya = gopi_nagoya - nagoya_all
diff_igs = gopi_igs - igs_all

mae_embrace = np.mean(np.abs(diff_embrace))
rmse_embrace = np.sqrt(np.mean(diff_embrace**2))

mae_maggia = np.mean(np.abs(diff_maggia))
rmse_maggia = np.sqrt(np.mean(diff_maggia**2))

mae_nagoya = np.mean(np.abs(diff_nagoya))
rmse_nagoya = np.sqrt(np.mean(diff_nagoya**2))

mae_igs = np.mean(np.abs(diff_igs))
rmse_igs = np.sqrt(np.mean(diff_igs**2))

print("="*70)
print("CONSOLIDATED RESULTS - December 18, 22, 19 and 1, 2024")
print("(28 measurements: 7 stations × 4 times)")
print("="*70)
print(f"GOPI vs EMBRACE:")
print(f"  MAE:  {mae_embrace:.2f}")
print(f"  RMSE: {rmse_embrace:.2f}")
print()
print(f"GOPI vs MAGGIA:")
print(f"  MAE:  {mae_maggia:.2f}")
print(f"  RMSE: {rmse_maggia:.2f}")
print()
print(f"GOPI vs NAGOYA:")
print(f"  MAE:  {mae_nagoya:.2f}")
print(f"  RMSE: {rmse_nagoya:.2f}")
print("="*70)

results_december = enhanced_analysis(data)
print_enhanced_results(results_december, "COMPREHENSIVE ANALYSIS - December 18, 22, 19 and 1, 2024")

CONSOLIDATED RESULTS - December 18, 22, 19 and 1, 2024
(28 measurements: 7 stations × 4 times)
GOPI vs EMBRACE:
  MAE:  6.40
  RMSE: 8.15

GOPI vs MAGGIA:
  MAE:  18.11
  RMSE: 18.73

GOPI vs NAGOYA:
  MAE:  10.20
  RMSE: 18.16
COMPREHENSIVE ANALYSIS - December 18, 22, 19 and 1, 2024

EMBRACE (Rank #1):
  MAE:                   6.40
  RMSE:                  8.15
  Pearson Correlation:   0.472 (p=0.011)
  Taylor Skill Score:    0.720
  Kling-Gupta Efficiency: 0.453
    - Correlation (r):   0.472
    - Alpha (σp/σo):     0.860
    - Beta (μp/μo):      1.034
  CRI Score:             0.807

MAGGIA (Rank #2):
  MAE:                   18.11
  RMSE:                  18.73
  Pearson Correlation:   0.830 (p=0.000)
  Taylor Skill Score:    0.915
  Kling-Gupta Efficiency: 0.690
    - Correlation (r):   0.830
    - Alpha (σp/σo):     1.021
    - Beta (μp/μo):      1.258
  CRI Score:             0.600

IGS (Rank #3):
  MAE:                   12.77
  RMSE:                  15.21
  Pearson Correlatio